# Fine-tuning the DeepGlobe U-Net on the annotated CBERS-4A tiles

Takes the frozen DeepGlobe checkpoint (EfficientNetB0 U-Net, 6 merged classes) and
fine-tunes it on the 50 hand-annotated 1024px CBERS-4A tiles exported from CVAT.

**The floor this run must beat.** The same checkpoint evaluated zero-shot on these
tiles (`notes/inpe_zero_shot_baseline.md`, 5% Unknown threshold, Unknown GT pixels
excluded from scoring):

| split | px acc | mIoU |
|---|---|---|
| all 50 tiles | 0.391 | 0.246 |
| test 10 tiles | 0.373 | 0.239 |

**The DeepGlobe reference (0.813 mIoU on its own validation set) is not a ceiling
for the number below and is not comparable to it.** Different dataset, different
sensor and radiometry, a different Unknown policy (DeepGlobe's reference excludes
Unknown as a class entirely, here it is one of the six trained classes), and a
different tiling grid. It is listed only because `report()` prints it.

**Two different mIoUs, do not confuse them.** `val_miou` in the wandb table is
Keras `MeanIoU(num_classes=6)`: it scores Unknown as a class and keeps every
sub-tile. The number comparable to the floor above is the one the evaluation cell
prints and logs as `protocol_miou_unk05` -- Unknown GT excluded, sub-tiles over
the threshold dropped. Unknown is 2.85% of val pixels but 0.02% of train pixels,
so the model can barely learn it and its IoU sits near zero, dragging `val_miou`
to roughly five-sixths of the protocol number.

**Decisions fixed for this run:**

* Unknown is an ordinary trained class (index 5) — no loss mask, no `sample_weight`.
* No cross-validation, deliberately: the splits were built for class
  representation on both sides, and scene-grouped folds would destroy that
  stratification. Train on `train`, validate on `test`, and `test` is used
  for validation only -- it never contributes a gradient.
* 4x4 non-overlapping 256px crops per 1024px tile (16 sub-tiles), no stride, no
  overlap. DeepGlobe's 3x3 stride-178 overlap grid does not apply here.
* `FREEZE_EPOCHS = 0` by default: the decoder is already pretrained on this exact
  taxonomy, so there is no random-head gradient shock to protect against, and
  `trainable=False` would pin the encoder's BatchNorm statistics to DeepGlobe's,
  forfeiting adaptation to CBERS radiometry. The mechanism is still there and
  switchable.

In [1]:
import importlib.metadata as md
import os
import subprocess
import sys

# Deliberately NOT silencing TF here (cell 2 does that). When TensorFlow is built
# with CUDA but finds no device, the reason is a dynamic-loader error on
# libcudart/libcudnn/libcublas -- and TF_CPP_MIN_LOG_LEVEL >= 2 hides exactly
# those lines, leaving only an unexplained "0 GPUs".
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '0'
import tensorflow as tf


def smi(query):
    try:
        return subprocess.run(['nvidia-smi', f'--query-gpu={query}', '--format=csv'],
                              capture_output=True, text=True, check=True).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError) as exc:
        return f'nvidia-smi unavailable: {exc}'


print(smi('index,name,memory.total,memory.free,driver_version'))

# The kernel's interpreter, not the shell's. A `pip list` run in a conda prompt
# describes a different environment than the kernel whenever the notebook was
# started from another env -- the usual reason TF sees no GPU while the nvidia
# wheels are demonstrably installed "somewhere".
nvidia = sorted(d.metadata['Name'] for d in md.distributions()
                if (d.metadata['Name'] or '').startswith('nvidia-'))
print(f'\nkernel python: {sys.executable}')
print(f'nvidia-* packages visible to THIS kernel: {len(nvidia)}')
print('  ' + (', '.join(n.replace('nvidia-', '') for n in nvidia) if nvidia else 'NONE'))

build = tf.sysconfig.get_build_info()
print(f'\ntf {tf.__version__} | built with CUDA: {tf.test.is_built_with_cuda()}'
      f' | built against CUDA {build.get("cuda_version")}'
      f' cuDNN {build.get("cudnn_version")}')

gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs visible to TensorFlow: {len(gpus)}')
for gpu in gpus:
    details = tf.config.experimental.get_device_details(gpu)
    cc = details.get('compute_capability')
    print(f'  {gpu.name}  {details.get("device_name", "?")}'
          + (f'  compute {cc[0]}.{cc[1]}' if cc else ''))
    tf.config.experimental.set_memory_growth(gpu, True)

if not gpus:
    # nvidia-smi seeing the card only proves the driver is fine. TF needs the
    # CUDA *user-space* libraries too, which ship as nvidia-* pip packages with
    # the tensorflow[and-cuda] extra.
    print(f'\ntensorflow loaded from: {tf.__file__}')
    if not nvidia:
        print('No nvidia-* packages in the kernel environment. Either TF was '
              'installed without its CUDA extra, or this kernel is not the env '
              'you ran pip in. Fix with, in THIS interpreter:\n'
              f'    {sys.executable} -m pip install '
              f'"tensorflow[and-cuda]=={tf.__version__}"')
    else:
        print('The CUDA wheels are present, so the failure is at load time. '
              'Read the loader errors printed above: they name the library '
              'that could not be opened, and the version it wanted.')
    raise RuntimeError('No GPU visible. This notebook is not meant to run on CPU.')


I0000 00:00:1789240039.703194 3538108 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


index, name, memory.total [MiB], memory.free [MiB], driver_version
0, NVIDIA GeForce RTX 3050 Laptop GPU, 4096 MiB, 3706 MiB, 580.173.02

kernel python: /home/jvpcms/ime/PFC/.venv/bin/python
nvidia-* packages visible to THIS kernel: 12
  cublas-cu12, cuda-cupti-cu12, cuda-nvcc-cu12, cuda-nvrtc-cu12, cuda-runtime-cu12, cudnn-cu12, cufft-cu12, curand-cu12, cusolver-cu12, cusparse-cu12, nccl-cu12, nvjitlink-cu12

tf 2.21.0 | built with CUDA: True | built against CUDA 12.5.1 cuDNN 9
GPUs visible to TensorFlow: 1
  /physical_device:GPU:0  NVIDIA GeForce RTX 3050 Laptop GPU  compute 8.6


In [2]:
import os
import random
import sys

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
sys.path.insert(0, 'train_scripts')

import numpy as np
import tensorflow as tf
import wandb
from dotenv import load_dotenv
from PIL import Image
from wandb.integration.keras import WandbMetricsLogger, WandbModelCheckpoint

from eval_inpe_tiles import (
    CLASS_NAMES,
    N_CLASSES,
    TILE,
    UNKNOWN_IDX,
    annotated_tiles,
    confusion,
    report,
    rgb_mask_to_labels,
)
from export_finetune_split import FINETUNE_DIR, export_split, scene_of
from train_deepglobe_unet import bce_dice_loss, check_gpu

load_dotenv()

from pathlib import Path

BASE_CHECKPOINT = Path('models/deepglobe_unet') / (
    'best-20260702-downsample-overlap-color-aug-dropout-30-merged-class'
    '-batch-32-lr-1e-4-miou-67.keras'
)
MODELS_DIR = Path('models/inpe_finetune')

LR              = 3e-5
BATCH_SIZE      = 8      # RTX 3050 4 GB — do not raise
EPOCHS          = 80
FREEZE_EPOCHS   = 0
UNFREEZE_LR_DIV = 4.0
ES_PATIENCE     = 15
LR_FACTOR       = 0.5
LR_PATIENCE     = 6
SEED            = 42

# Zero-shot floor on the test split, per Unknown threshold, from
# notes/inpe_zero_shot_baseline.md. Keyed by threshold so each result is compared
# against the floor measured under the same cutoff.
ZERO_SHOT_TEST = {0.05: (0.373, 0.239), 0.25: (0.403, 0.252)}

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

check_gpu()
print(f'tf {tf.__version__} | checkpoint {BASE_CHECKPOINT.name}')

GPU(s) available: ['/physical_device:GPU:0']
TensorFlow built with CUDA: True
tf 2.21.0 | checkpoint best-20260702-downsample-overlap-color-aug-dropout-30-merged-class-batch-32-lr-1e-4-miou-67.keras


## Data

`export_finetune_split()` copies each annotated tile and its CVAT mask into
`annotation_tiles/finetune/<split>/{images,masks}/`. It is idempotent and never
touches `half_labels/`, so it stays re-runnable when the second annotation batch
lands.

Each 1024px pair is then cut into 16 non-overlapping 256px sub-tiles and the mask
decoded to class indices with `rgb_mask_to_labels`. 640 train + 160 test sub-tiles
is ~126 MB of uint8, so it is held in memory and fed to `tf.data` with
`from_tensor_slices` — no `.npy` tiles on disk.

In [3]:
written = export_split()
for split, tile_ids in written.items():
    print(f'{split:<6} {len(tile_ids):>3} tiles  '
          f'{len({scene_of(t) for t in tile_ids}):>3} scenes')


def load_split(split):
    '''(N,256,256,3) uint8 images + (N,256,256) int32 labels for one split.

    Iterates the tile ids `export_split` just resolved rather than globbing the
    output directory. Globbing would trust whatever is on disk, so a tile left
    over from an earlier split assignment would be silently trained on -- and
    the guard against exactly that lives in `annotated_tiles`, upstream of here.
    '''
    images, labels = [], []
    for tile_id in sorted(written[split]):
        img_path = FINETUNE_DIR / split / 'images' / f'{tile_id}.png'
        mask_path = FINETUNE_DIR / split / 'masks' / f'{tile_id}.png'
        img = np.array(Image.open(img_path).convert('RGB'))
        lbl = rgb_mask_to_labels(np.array(Image.open(mask_path).convert('RGB')))
        if img.shape[:2] != lbl.shape:
            raise ValueError(f'{img_path.stem}: image {img.shape[:2]} != mask {lbl.shape}')
        h, w = lbl.shape
        if h % TILE or w % TILE:
            raise ValueError(f'{img_path.stem}: {h}x{w} is not a multiple of {TILE}')
        for r in range(0, h, TILE):
            for c in range(0, w, TILE):
                images.append(img[r:r + TILE, c:c + TILE])
                labels.append(lbl[r:r + TILE, c:c + TILE])
    return np.stack(images), np.stack(labels).astype(np.int32)


def print_distribution(name, labels):
    counts = np.bincount(labels.ravel(), minlength=N_CLASSES)
    print(f'{name} class distribution')
    for class_name, count in zip(CLASS_NAMES, counts):
        print(f'  {class_name:<22} {count / counts.sum():6.2%}')


x_train, y_train = load_split('train')
x_val, y_val = load_split('test')

# Derived from the tile counts, not hard-coded: these must keep holding when the
# second annotation batch lands and the splits grow.
SUB_TILES = (1024 // TILE) ** 2
for name, arr, n_tiles in (
    ('x_train', x_train, len(written['train'])),
    ('x_val', x_val, len(written['test'])),
):
    assert arr.shape == (n_tiles * SUB_TILES, TILE, TILE, 3), (name, arr.shape)
assert y_train.shape == x_train.shape[:3], y_train.shape
assert y_val.shape == x_val.shape[:3], y_val.shape
assert x_train.dtype == np.uint8 and x_val.dtype == np.uint8
for y in (y_train, y_val):
    assert y.min() >= 0 and y.max() < N_CLASSES, (y.min(), y.max())

print(f'\ntrain {x_train.shape} {x_train.nbytes / 1e6:.0f} MB | '
      f'val {x_val.shape} {x_val.nbytes / 1e6:.0f} MB')
print_distribution('train', y_train)
print_distribution('val', y_val)

train   40 tiles   18 scenes
test    10 tiles    9 scenes

train (640, 256, 256, 3) 126 MB | val (160, 256, 256, 3) 31 MB
train class distribution
  Urban                   4.86%
  Agriculture_Rangeland  42.64%
  Forest                 11.51%
  Water                  36.09%
  Barren                  4.88%
  Unknown                 0.02%
val class distribution
  Urban                  14.48%
  Agriculture_Rangeland  30.23%
  Forest                  7.17%
  Water                  44.16%
  Barren                  1.11%
  Unknown                 2.85%


In [4]:
def augment_fn(img, label):
    '''Verbatim copy of the augmentation nested in make_datasets() of
    train_scripts/train_deepglobe_unet.py — it is part of the recipe being
    carried over, and changing it would add a third variable to the comparison.
    '''
    img_f    = tf.cast(img, tf.float32)
    combined = tf.concat([img_f, tf.cast(tf.expand_dims(label, -1), tf.float32)], axis=-1)
    combined = tf.image.random_flip_left_right(combined)
    combined = tf.image.random_flip_up_down(combined)
    k        = tf.random.uniform((), minval=0, maxval=4, dtype=tf.int32)
    combined = tf.image.rot90(combined, k)
    img      = tf.cast(combined[:, :, :3], tf.uint8)
    label    = tf.cast(combined[:, :, 3], tf.int32)
    # colour jitter applied to image only (label unchanged)
    img_f = tf.cast(img, tf.float32) / 255.0
    img_f = tf.image.random_brightness(img_f, max_delta=0.2)
    img_f = tf.image.random_contrast(img_f, lower=0.8, upper=1.2)
    img_f = tf.image.random_saturation(img_f, lower=0.8, upper=1.2)
    img_f = tf.image.random_hue(img_f, max_delta=0.05)
    img   = tf.cast(tf.clip_by_value(img_f * 255.0, 0, 255), tf.uint8)
    return img, label


train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(len(x_train), reshuffle_each_iteration=True)
    .map(augment_fn, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
print(f'Train batches: {len(train_ds)} | Val batches: {len(val_ds)}')

I0000 00:00:1789240080.282989 3538108 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2091 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6
W0000 00:00:1789240080.284823 3538108 cpu_allocator_impl.cc:82] Allocation of 125829120 exceeds 10% of free system memory.
W0000 00:00:1789240080.446652 3538108 cpu_allocator_impl.cc:82] Allocation of 167772160 exceeds 10% of free system memory.
W0000 00:00:1789240080.586451 3538108 cpu_allocator_impl.cc:82] Allocation of 125829120 exceeds 10% of free system memory.


Train batches: 80 | Val batches: 20


## Model

`compile=False` on `load_model` avoids having to pass `custom_objects` for
`bce_dice_loss`.

**Freeze mechanism.** `build_unet()` passes `input_tensor=inputs` and taps the skip
connections via `backbone.get_layer(n).output`, so the encoder is flattened into the
outer functional graph — after `load_model` there is no nested backbone submodel to
toggle. The encoder is therefore identified by layer name: EfficientNetB0's layers
are the ones prefixed `stem_`, `blockN...`, or `top_`. The match count is printed so
a silent no-match is visible.

**`DROPOUT` is deliberately not a notebook constant.** The rate lives in the
checkpoint's `Dropout` layers, so setting one here would change nothing while
wandb advertised it as a run parameter. The cell below reads the real rate off the
loaded model and logs that. Changing it means rebuilding the model, not editing a
constant.


In [5]:
model = tf.keras.models.load_model(BASE_CHECKPOINT, compile=False)
if model.output_shape[-1] != N_CLASSES:
    raise ValueError(f'model outputs {model.output_shape[-1]} classes, expected {N_CLASSES}')


def compile_model(lr):
    per_class_iou = [
        tf.keras.metrics.IoU(
            num_classes=N_CLASSES,
            target_class_ids=[i],
            name=f'iou_{CLASS_NAMES[i].lower()}',
            sparse_y_pred=False,
        )
        for i in range(N_CLASSES)
    ]
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=bce_dice_loss,
        metrics=[
            tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy'),
            tf.keras.metrics.MeanIoU(num_classes=N_CLASSES, name='miou',
                                     sparse_y_pred=False),
            *per_class_iou,
        ],
    )


ENCODER_PREFIXES = ('stem_', 'top_') + tuple(f'block{i}' for i in range(1, 8))


def encoder_layers():
    return [l for l in model.layers if l.name.startswith(ENCODER_PREFIXES)]


def set_encoder_trainable(trainable):
    layers = encoder_layers()
    for layer in layers:
        layer.trainable = trainable
    print(f'encoder layers matched: {len(layers)} / {len(model.layers)} '
          f'-> trainable={trainable}')


# Read the decoder dropout off the loaded model rather than declaring it as a
# constant. The rate is baked into the checkpoint's layers, so a notebook-level
# DROPOUT would be inert -- and wandb would advertise a run parameter that never
# touched the run. To actually change it you have to rebuild the model.
# Only the decoder's Dropout layers: EfficientNetB0 carries its own
# stochastic-depth Dropout at ten different rates, so filtering by type alone
# picks those up too.
dropout_rates = {
    l.rate for l in model.layers
    if isinstance(l, tf.keras.layers.Dropout)
    and not l.name.startswith(ENCODER_PREFIXES)
}
if len(dropout_rates) != 1:
    raise ValueError(f'expected one decoder dropout rate, found {dropout_rates}')
DECODER_DROPOUT = dropout_rates.pop()

compile_model(LR)
print(f'Total params: {sum(tf.size(w).numpy() for w in model.weights):,}')
print(f'encoder layers matched: {len(encoder_layers())} / {len(model.layers)}')
print(f'decoder dropout (from the checkpoint): {DECODER_DROPOUT}')

Total params: 11,297,161
encoder layers matched: 234 / 288
decoder dropout (from the checkpoint): 0.3


## wandb

The config mirrors `train_deepglobe_unet.py`'s field-for-field so the base run and
this one line up in a single wandb table.

In [6]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
run_name = f'finetune-inpe-effb0-bs{BATCH_SIZE}-lr{LR:g}'

config = dict(
    epochs          = EPOCHS,
    lr              = LR,
    lr_factor       = LR_FACTOR,
    lr_patience     = LR_PATIENCE,
    es_patience     = ES_PATIENCE,
    batch_size      = BATCH_SIZE,
    encoder         = 'efficientnetb0',
    input_shape     = (TILE, TILE, 3),
    n_classes       = N_CLASSES,
    loss            = 'bce_dice',
    optimizer       = 'adam',
    augmentation    = 'hflip+vflip+rot90+brightness+contrast+saturation+hue',
    decoder_dropout = DECODER_DROPOUT,
    tiling          = '4x4 non-overlapping grid (256px)',
    freeze_epochs   = FREEZE_EPOCHS,
    unfreeze_lr_div = UNFREEZE_LR_DIV,
    class_merge     = 'Agriculture+Rangeland merged at load time (7 -> 6 classes)',
    base_checkpoint = BASE_CHECKPOINT.name,
    finetune        = True,
    dataset         = 'inpe_cbers_annotation_tiles_50',
    n_train_tiles   = len(written['train']),
    n_val_tiles     = len(written['test']),
    n_train_scenes  = len({scene_of(t) for t in written['train']}),
    n_val_scenes    = len({scene_of(t) for t in written['test']}),
    unknown_policy  = 'trained and scored as an ordinary class',
)

wandb.init(project='pitcic-segmentation', name=run_name, config=config)

callbacks = [
    WandbMetricsLogger(log_freq='epoch'),
    WandbModelCheckpoint(
        str(MODELS_DIR / 'best.keras'),
        monitor='val_miou',
        mode='max',
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_miou',
        mode='max',
        factor=LR_FACTOR,
        patience=LR_PATIENCE,
        min_lr=1e-7,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_miou',
        mode='max',
        patience=ES_PATIENCE,
        restore_best_weights=True,
        verbose=1,
    ),
]
print(run_name)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: jvpcms (jvpcms-ime) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: WARNING When using `save_best_only`, ensure that the `filepath` argument contains formatting placeholders like `{epoch:02d}` or `{batch:02d}`. This ensures correct interpretation of the logged artifacts.


finetune-inpe-effb0-bs8-lr3e-05


wandb: WARNING Artifact "run_e22t2ndl_model" already exists with the same content. No new version will be created.
wandb: WARNING Artifact "run_e22t2ndl_model" already exists with the same content. No new version will be created.
wandb: WARNING Artifact "run_e22t2ndl_model" already exists with the same content. No new version will be created.
wandb: WARNING Artifact "run_e22t2ndl_model" already exists with the same content. No new version will be created.


In [7]:
if FREEZE_EPOCHS > 0:
    # Phase 1: decoder-only warmup. Off by default — see the header cell.
    set_encoder_trainable(False)
    compile_model(LR)
    print(f'Phase 1: encoder frozen for {FREEZE_EPOCHS} epochs')
    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=FREEZE_EPOCHS,
        callbacks=[WandbMetricsLogger(log_freq='epoch')],
    )

    # Phase 2: recompile required after a trainable change; the reduced LR
    # protects the pretrained encoder features right after unfreeze.
    set_encoder_trainable(True)
    compile_model(LR / UNFREEZE_LR_DIV)
    print(f'Phase 2: encoder unfrozen — full training at lr={LR / UNFREEZE_LR_DIV:g}')

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    initial_epoch=FREEZE_EPOCHS,
    callbacks=callbacks,
)

model.save(MODELS_DIR / 'final.keras')
print(f'Model saved to {MODELS_DIR}')
# wandb.finish() deliberately deferred to the evaluation cell: the protocol mIoU
# is the number comparable to the 0.239 floor, and it has to reach the run.

Epoch 1/80


W0000 00:00:1789240086.744205 3538108 cpu_allocator_impl.cc:82] Allocation of 125829120 exceeds 10% of free system memory.
W0000 00:00:1789240086.815150 3538108 cpu_allocator_impl.cc:82] Allocation of 167772160 exceeds 10% of free system memory.
I0000 00:00:1789240113.619485 3539481 service.cc:153] XLA service 0x7e5c840cb610 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1789240113.619539 3539481 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3050 Laptop GPU, Compute Capability 8.6 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.22.0)
I0000 00:00:1789240114.571924 3539481 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
W0000 00:00:1789240116.517505 3539481 assert_op.cc:39] Ignoring Assert operator compile_loss/bce_dice_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
I0000 00:00:1789240118.784862 3

80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step - accuracy: 0.5396 - iou_agriculture_rangeland: 0.6049 - iou_barren: 0.1246 - iou_forest: 0.1917 - iou_unknown: 9.4660e-05 - iou_urban: 0.2166 - iou_water: 0.2319 - loss: 2.9485 - miou: 0.2289

W0000 00:00:1789240270.812244 3539480 assert_op.cc:39] Ignoring Assert operator compile_loss/bce_dice_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert



Epoch 1: val_miou improved from None to 0.45303, saving model to models/inpe_finetune/best.keras

Epoch 1: finished saving model to models/inpe_finetune/best.keras
80/80 ━━━━━━━━━━━━━━━━━━━━ 193s 287ms/step - accuracy: 0.6051 - iou_agriculture_rangeland: 0.6037 - iou_barren: 0.1347 - iou_forest: 0.2847 - iou_unknown: 5.1099e-05 - iou_urban: 0.2678 - iou_water: 0.3949 - loss: 2.5316 - miou: 0.2810 - val_accuracy: 0.8166 - val_iou_agriculture_rangeland: 0.7036 - val_iou_barren: 0.1609 - val_iou_forest: 0.4000 - val_iou_unknown: 6.3857e-04 - val_iou_urban: 0.5864 - val_iou_water: 0.8667 - val_loss: 1.7312 - val_miou: 0.4530 - learning_rate: 3.0000e-05
Epoch 2/80
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step - accuracy: 0.7220 - iou_agriculture_rangeland: 0.6259 - iou_barren: 0.1860 - iou_forest: 0.4850 - iou_unknown: 0.0000e+00 - iou_urban: 0.4230 - iou_water: 0.6398 - loss: 1.8407 - miou: 0.3945
Epoch 2: val_miou improved from 0.45303 to 0.52371, saving model to models/inpe_finetune/best.ker

## Evaluation

Same protocol as the zero-shot baseline so the numbers are directly comparable:
sub-tiles at or above the Unknown threshold are dropped, remaining Unknown GT
pixels are excluded from scoring, then `confusion()` + `report()` from
`eval_inpe_tiles`.

Prediction runs in chunks via `model(chunk, training=False)` rather than
`model.predict`: `predict` concatenates every `(256,256,6)` softmax map before
returning, which is the largest allocation in the run and exhausts the 4 GB card.

In [8]:
def predict_labels(images, batch_size=BATCH_SIZE):
    preds = np.empty(images.shape[:3], dtype=np.int32)
    for start in range(0, len(images), batch_size):
        chunk = images[start:start + batch_size].astype(np.float32)
        preds[start:start + len(chunk)] = np.argmax(
            model(chunk, training=False).numpy(), axis=-1
        )
    return preds


val_preds = predict_labels(x_val)
val_unknown_frac = (y_val == UNKNOWN_IDX).mean(axis=(1, 2))

for thresh in (0.05, 0.25):
    keep = val_unknown_frac < thresh
    print(f"\n{'=' * 78}\nUnknown threshold {thresh:.0%}: "
          f'{int(keep.sum())} / {len(keep)} sub-tiles kept')
    if not keep.any():
        print('  every sub-tile is above the threshold -- nothing to score')
        continue
    lab, prd = y_val[keep], val_preds[keep]
    valid = lab != UNKNOWN_IDX
    print(f'  {valid.mean():.1%} of pixels scored '
          f'({int((~valid).sum()):,} Unknown px excluded)')

    cm = confusion(lab[valid], prd[valid])
    report(cm)

    # The protocol number, logged under its own key. `val_miou` in the wandb
    # table is Keras MeanIoU over all 6 classes with Unknown scored and every
    # sub-tile kept -- a different measurement that must not be read against the
    # zero-shot floor.
    inter = np.diag(cm).astype(float)
    union = cm.sum(0) + cm.sum(1) - np.diag(cm)
    present = cm.sum(1) > 0
    iou = np.where(union > 0, inter / np.maximum(union, 1), np.nan)
    px_acc = np.trace(cm) / cm.sum()
    miou = np.nanmean(np.where(present, iou, np.nan))

    floor_acc, floor_miou = ZERO_SHOT_TEST[thresh]
    print(f'\n  zero-shot floor at this threshold: '
          f'px acc {floor_acc:.3f} / mIoU {floor_miou:.3f}')
    print(f'  fine-tuned:                        '
          f'px acc {px_acc:.3f} / mIoU {miou:.3f}')
    print(f'  delta:                             '
          f'{px_acc - floor_acc:+.3f} / {miou - floor_miou:+.3f}')

    if wandb.run is not None:
        tag = f'{int(thresh * 100):02d}'
        wandb.run.summary[f'protocol_px_acc_unk{tag}'] = float(px_acc)
        wandb.run.summary[f'protocol_miou_unk{tag}'] = float(miou)
        wandb.run.summary[f'protocol_miou_gain_unk{tag}'] = float(miou - floor_miou)

if wandb.run is not None:
    wandb.finish()

W0000 00:00:1789240623.458876 3538108 bfc_allocator.cc:311] Allocator (GPU_0_bfc) ran out of memory trying to allocate 784.00MiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
W0000 00:00:1789240623.458919 3538108 bfc_allocator.cc:311] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.41GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
W0000 00:00:1789240623.702917 3538108 bfc_allocator.cc:311] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.03GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
W0000 00:00:1789240623.891366 3538108 bfc_allocator.cc:311] Allocator (GPU_0_bfc) ran out of memory trying to allocate 2.16GiB with freed_by


Unknown threshold 5%: 144 / 160 sub-tiles kept
  100.0% of pixels scored (0 Unknown px excluded)

Pixel accuracy: 0.9377
class                       IoU        GT px      pred px
Urban                    0.8883    1,518,083    1,439,136
Agriculture_Rangeland    0.8153    2,426,034    2,773,202
Forest                   0.7555      746,054      700,175
Water                    0.9558    4,630,938    4,444,104
Barren                   0.5009      116,075       80,567
Unknown                     n/a            0            0

mIoU over classes present in GT: 0.7832
(DeepGlobe val reference, 5-class excl. Unknown: 0.813)
pred = Unknown on labeled px: 0.00%

Row-normalized confusion (recall per GT class)
                            Urban Agricultur     Forest      Water     Barren    Unknown
Urban                        0.92       0.07       0.01       0.00       0.00       0.00
Agriculture_Rangeland        0.01       0.96       0.02       0.00       0.00       0.00
Forest                  

epoch/accuracy,▁▄▅▅▅▅▆▆▆▇▇▇▇▇▇▇▇▇███▇█████
epoch/epoch,▁▁▂▂▂▂▃▃▃▃▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇██
epoch/iou_agriculture_rangeland,▁▂▃▃▄▃▅▅▅▆▆▆▆▆▇▇▇▇███▇█████
epoch/iou_barren,▁▂▂▂▂▂▃▂▃▄▄▄▅▄▆▅▆▆▇▇█▇▇▇█▇▇
epoch/iou_forest,▁▄▅▅▆▆▇▆▆▇▇▇▇▇▇▇█▇▇█▇▇▇██▇█
epoch/iou_unknown,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/iou_urban,▁▃▃▄▄▄▅▆▆▇▇▇▇▆▇▇▇▇▇▇█▇▇█▇▇█
epoch/iou_water,▁▄▅▅▆▆▇▆▆▇▇▇▇▇▇▇█████▇█████
epoch/learning_rate,██████████████████▃▃▃▃▃▃▁▁▁
epoch/loss,█▄▄▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▂▁▁▁▁▁
+10,...


## How to read the number above

**No cross-validation, by design.** The train and test splits were built so that
every class has meaningful representation on both sides. Scene-grouped k-fold
would re-cut the tiles by scene and destroy that stratification: with 19 scenes
and several of them single-class (three are solid open water), a grouped fold
can easily hold out a partition in which a class is absent from training, from
validation, or from both. The resulting per-fold mIoU would swing on which
scenes landed where rather than on the model. A fixed split that is balanced on
purpose is the more informative estimator here, and `test` is used for
validation only -- it never contributes a gradient.

**What the score is.** `val_miou` gates both `best.keras` and early stopping, so
the reported figure is a *validation* score on the tiles that also drove model
selection, not a held-out test score. Report it as such. The honest framing is
"the fine-tuned model reaches X on the validation split, against a 0.239
zero-shot floor on the same tiles" -- the comparison to the floor is exact,
since both are measured on the identical tiles under the identical protocol.

**The two mIoUs are still different quantities.** `val_miou` in wandb is Keras
`MeanIoU(num_classes=6)`, scoring Unknown and keeping all 160 sub-tiles.
`protocol_miou_unk05` is the one comparable to the zero-shot floor: Unknown GT
excluded, sub-tiles over the threshold dropped. Unknown is 2.85% of val pixels
against 0.02% of train pixels, so its IoU sits near zero and drags `val_miou` to
roughly five-sixths of the protocol number.
